# 13 - End-to-end evaluation: baselines, deployed decision rule, ablations
Plan items F1-F3. Every strategy replays the **same test sequences** through the same loop (`src/controller/replay.py`), using the cached YOLO11n / YOLO11s outputs of notebook 07, so differences come from the strategy only. **Nothing is tuned here**: every setting comes from earlier val choices (notebooks 07-11) and the repo configs; test is reported once.

**Two decision pipelines** for the per-slot state:
- **nb07 rule** (as in notebook 10): occupied when the slot score >= the notebook 07 score threshold (max val macro-F1), no temporal filter.
- **deployed rule** (configs, cost ratio C_FV/C_FO = 5): isotonic P(occupied) > 1/6 (notebook 11), then the asymmetric temporal filter (flip to occupied at once, to vacant after 3 frames, immediately when the margin >= 0.75). The probability rule is applied as its exact score equivalent per model. Notebook 11 tested the two parts **separately**; this is the first test of the combination, inside the adaptive controller.

**Strategies:** fixed YOLO11n, fixed YOLO11s, periodic YOLO11n (c=2), scene-triggered without slot risk, AdaptivePark (accuracy and efficiency points of notebook 10), oracle (labels; upper bound).
**Ablations** of AdaptivePark under the deployed rule: each D feature with weight > 0 dropped (weights re-normalised, the enhanced threshold re-derived as the **same val quantile** of the new D), slot risk dropped (= scene-triggered), critical-slot refresh dropped, max-age refresh dropped, temporal filter dropped, calibration dropped. Hysteresis is 0 / dwell 1 in the chosen setting, so "no hysteresis" is the setting itself.
**Metrics:** accuracy and macro-F1 over every frame (skipped frames judged on the state they show), FV / FO, **cost per 1000 slot decisions at C_FV/C_FO = 5**, compute per frame in YOLO11n units (YOLO11s = 2.63, a Kaggle T4 ratio, **not a Jetson number**), false state flips per 1000 slot-frames, transition detection and delay. Key differences get a **paired bootstrap over camera-day sequences** (95% interval).

**Inputs (Add Input -> Your Work):** notebook 03 output (`artifacts/splits/`), notebook 07 output (full run), notebook 09 output (full run), notebook 10 output (full run, `controller_record.json`), notebook 11 output (full run, `cost_sensitive_record.json`).
**Accelerator:** none (CPU). Internet ON only to clone the repo.
**How to run:** `SMOKE = True` + Run All (a few sequences). Then `SMOKE = False` + **Save Version -> Save & Run All**.

In [ ]:
# --- Setup: get the project code onto Kaggle ---
# Preferred: attach the code as a Kaggle dataset (works for a private GitHub repo).
# Fallback: git clone (only works if the GitHub repo is public and Internet is ON).
import os, sys, glob, shutil, subprocess
REPO_URL = "https://github.com/Amith2277/Edge-Computing-EndSem.git"
REPO_DIR = "/kaggle/working/repo"
MARK = os.path.join("src", "common", "config.py")

def find_attached_code():
    for pat in ("/kaggle/input/*/", "/kaggle/input/*/*/", "/kaggle/input/*/*/*/"):
        for d in sorted(glob.glob(pat)):
            if os.path.isfile(os.path.join(d, MARK)):
                return d.rstrip("/")
    return None

if os.path.isdir(REPO_DIR):
    print("using existing", REPO_DIR)
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):  # cloned earlier: fetch the latest code
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout.strip())
else:
    attached = find_attached_code()
    if attached:
        shutil.copytree(attached, REPO_DIR)
        print("copied attached code dataset:", attached)
    else:
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout)
        if r.returncode != 0:
            raise SystemExit("No attached code dataset found and git clone failed (private repo?). "
                             "Upload adaptivepark-code.zip as a Kaggle dataset and Add Input it, then re-run.")
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("code at", REPO_DIR, "| files:", sorted(os.listdir(REPO_DIR))[:12])


In [ ]:
# Settings and inputs
SMOKE = True                      # True: a few sequences per split. False: every val and test sequence
SMOKE_SEQUENCES = 8
LIGHT, HEAVY = "yolo11n", "yolo11s"
N_BOOT = 2000                     # paired bootstrap resamples (camera-day sequences)
TOL_FRAMES = 3                    # a true state change counts as detected within this many frames (as notebook 08)
MAX_GAP_FACTOR = 3.0              # same sequences as notebooks 08-12
INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"

import os, json, time, csv, gzip, pickle, copy
from collections import defaultdict
from datetime import datetime
from pathlib import Path
import numpy as np
from src.common.config import load_yaml, config_hash
from src.common.status import new_run_id, write_status
from src.training import yolo_train as yt
from src.evaluation import det_eval as de, bootstrap as bs
from src.occupancy import association as asc, cost_sensitive as cs
from src.temporal import filters as tf
from src.scene_analysis import features as sf
from src.slot_risk import risk as rk
from src.controller import replay as rp
CCFG, OCFG, TCFG = load_yaml("configs/adaptive_controller.yaml"), load_yaml("configs/occupancy.yaml"), load_yaml("configs/temporal.yaml")

def find_one(name, folder, what):
    p = [x for x in yt.find_files(INPUT_ROOT, name) if x.parent.name == folder]
    assert p, "{} not found: attach {}".format(name, what)
    return p[0]
man = find_one("split_manifest.csv", "splits", "notebook 03's output")
ROW_BY_ID = {int(r["image_id"]): r for r in de.read_manifest(man)}
SPLIT_INFO = json.loads((man.parent / "split_info.json").read_text()); H = SPLIT_INFO["manifest_hash"]
OCC = find_one("slot_scores_test.csv.gz", "occupancy", "notebook 07's output (full run)").parent
st07 = json.loads((OCC / "status.json").read_text()); assert st07["state"] == "COMPLETE"
R09p = find_one("scene_risk_record.json", "scene_risk", "notebook 09's output (full run)"); SCN = R09p.parent; R09 = json.loads(R09p.read_text())
R10 = json.loads(find_one("controller_record.json", "controller", "notebook 10's output (full run)").read_text())
R11 = json.loads(find_one("cost_sensitive_record.json", "cost_sensitive", "notebook 11's output (full run)").read_text())
for name, r in (("09", R09), ("10", R10), ("11", R11)):
    assert not r["smoke_test"] and r["dataset_split_hash"] == H and r["notebook07_run_id"] == st07["run_id"], \
        "notebook {} output is a smoke run or does not match this split / notebook 07 run".format(name)
assert R10["notebook09_run_id"] == R09["run_id"], "notebook 10 used a different notebook 09 run"

THR07, ASSOC = {}, {}
for v in (LIGHT, HEAVY):
    rec = json.loads((OCC / "occupancy_record_{}.json".format(v)).read_text())
    assert not rec["smoke_test"] and rec["dataset_split_hash"] == H, v
    THR07[v] = float(rec["chosen_on_val"]["conf_thr"]); ASSOC[v] = rec["chosen_on_val"]
# deployed decision rule from the configs, calibrators from notebook 11
DEC = OCFG["decision"]
C_FV, C_FO = float(DEC["cost_false_vacant"]), float(DEC["cost_false_occupied"])
P_THR = cs.bayes_threshold(C_FV, C_FO)
METH = DEC["calibration"]["method"]
CAL, T_ISO = {}, {}
for v in (LIGHT, HEAVY):
    assert R11["calibration"][v]["chosen_by_val_log_loss"] == METH, "configs/occupancy.yaml calibrator differs from notebook 11's val choice for " + v
    CAL[v] = cs.from_params(R11["calibration"][v]["params"][METH])
    T_ISO[v] = cs.score_threshold_for(CAL[v], P_THR)
FILT = {v: {k: x for k, x in TCFG["deployed"][v].items()} for v in (LIGHT, HEAVY)}
assert FILT[LIGHT] == FILT[HEAVY], "the replay applies one temporal filter to both models"
FILT = FILT[LIGHT]
COST = {k: float(x) for k, x in R10["cost_units"].items()}
W_D = np.array([R09["difficulty"]["weights"][f] for f in sf.FEATURES]); SCALES = R09["difficulty"]["scales"]
W_R = np.array([R09["slot_risk"]["weights"][c] for c in rk.COMPONENTS]); AGG = R09["slot_risk"]["aggregation"]
BASECTX = {"wD": W_D, "wR": W_R, "agg": AGG, "age_ref_s": float(R09["slot_risk"]["max_age_s_for_A"]),
           "inst_frames": int(R09["instability_frames"]), "occ_scale": float(SCALES["occlusion"]), "light": LIGHT, "heavy": HEAVY}
PIPES = {"nb07 rule": {"thr": THR07, "filter": None, "fixable_key": "fixable"},
         "isotonic only": {"thr": T_ISO, "filter": None, "fixable_key": "fixable_iso"},
         "filter only": {"thr": THR07, "filter": FILT, "fixable_key": "fixable"},
         "deployed": {"thr": T_ISO, "filter": FILT, "fixable_key": "fixable_iso"}}
CHOSEN = R10["chosen"]
SLOTS = {s["slot_id"]: s for s in json.loads((OCC / "slots.json").read_text())}
print("nb07 score thresholds", THR07, "| deployed: {} P(O) > {:.4f} (C_FV/C_FO = {:g}) = score >= {}".format(METH, P_THR, C_FV / C_FO, {k: round(x, 4) for k, x in T_ISO.items()}))
print("temporal filter", tf.label(FILT), "| compute units", COST, "(Kaggle T4 ratio, not Jetson)")
print("controller (notebook 10, accuracy point):", {k: CHOSEN["adaptive"]["accuracy_point"][k] for k in ("d_hi", "r_hi", "r_crit", "d_lo", "margin", "min_dwell", "max_age_s")})
OUT = Path(WORK) / "artifacts" / "evaluation"
OUT.mkdir(parents=True, exist_ok=True)
SUF = "_smoke" if SMOKE else ""
CFG_HASH = config_hash({"smoke": SMOKE, "smoke_sequences": SMOKE_SEQUENCES, "boot": N_BOOT, "tol": TOL_FRAMES, "gap": MAX_GAP_FACTOR, "split": H,
                        "nb07": st07["run_id"], "nb09": R09["run_id"], "nb10": R10["run_id"], "nb11": R11["run_id"],
                        "decision": DEC, "filter": FILT, "controller": CCFG})
RUN_ID = new_run_id("evl")
STATUS = OUT / "status.json"
write_status(STATUS, "13_end_to_end_evaluation", RUN_ID, "RUNNING", CFG_HASH, "inputs loaded")

In [ ]:
# Per-frame inputs for both models (as notebook 10), plus "fixable" slots under each decision threshold
def parse_ts(s):
    return datetime.fromisoformat(s.replace("Z", "")).timestamp()

def load_scores(split):
    cols = defaultdict(list)
    with gzip.open(str(OCC / "slot_scores_{}.csv.gz".format(split)), "rt") as f:
        for r in csv.DictReader(f):
            for k in ("image", "camera", "timestamp", "slot_id"):
                cols[k].append(r[k])
            cols["image_id"].append(int(r["image_id"])); cols["gt"].append(int(r["gt_occupied"])); cols["valid"].append(int(r["valid"]))
            for v in (LIGHT, HEAVY):
                cols[v].append(float(r["score_" + v]))
    return {k: np.array(x) for k, x in cols.items()}

PREP = {}
def build(split):
    o = load_scores(split)
    with gzip.open(str(SCN / "frame_features_{}.csv.gz".format(split)), "rt") as f:
        ff = {r["image"]: r for r in csv.DictReader(f)}
    dets = {}
    for v in (LIGHT, HEAVY):
        z = np.load(str(OCC / "detections" / "{}_{}.npz".format(v, split)))
        dets[v] = {"dets": z["dets"], "starts": z["starts"], "row": {str(n): i for i, n in enumerate(z["names"])}}
    names, first, counts = np.unique(o["image"], return_index=True, return_counts=True)
    fr = np.argsort(first); names, first, counts = names[fr], first[fr], counts[fr]
    cam = o["camera"][first]; iid = o["image_id"][first]
    sec = np.array([parse_ts(t) for t in o["timestamp"][first]]); day = np.array([ROW_BY_ID[int(i)]["day"] for i in iid])
    sp = tf.split_sequences(cam, sec, day, MAX_GAP_FACTOR)
    cam_cols = defaultdict(dict)
    for c, s in zip(o["camera"], o["slot_id"]):
        cam_cols[c].setdefault(s, len(cam_cols[c]))
    ids = sorted(set(sp["seq"].tolist()))
    if SMOKE:
        ids = [ids[i] for i in np.linspace(0, len(ids) - 1, min(SMOKE_SEQUENCES, len(ids))).astype(int)]
    seqs = []
    for q in ids:
        fidx = np.nonzero(sp["seq"] == q)[0]; fidx = fidx[np.argsort(sec[fidx], kind="stable")]
        c = cam[fidx[0]]; r0 = ROW_BY_ID[int(iid[fidx[0]])]
        S = {"camera": c, "site": r0["site"], "day": day[fidx[0]], "sec": sec[fidx], "n_cols": len(cam_cols[c]), "names": [str(names[f]) for f in fidx],
             "image_ids": iid[fidx], "cols": [], "valid": [], "gt": [], "fixable": [], "fixable_iso": [], "B": [], "M": [],
             "out": {v: {"score": [], "detf": [], "occ": []} for v in (LIGHT, HEAVY)}}
        for f in fidx:
            a, b = first[f], first[f] + counts[f]
            sids = tuple(o["slot_id"][a:b]); valid = o["valid"][a:b].astype(bool); gt = o["gt"][a:b].astype(bool)
            S["cols"].append(np.array([cam_cols[c][s] for s in sids])); S["valid"].append(valid); S["gt"].append(o["gt"][a:b])
            if sids not in PREP:
                PREP[sids] = asc.prepare_slots([SLOTS[s]["polygon"] for s in sids])
            for v in (LIGHT, HEAVY):
                D = dets[v]; k = D["row"][str(names[f])]; dd = D["dets"][D["starts"][k]:D["starts"][k + 1]]
                pairs = asc.image_pairs(PREP[sids], dd)
                S["out"][v]["score"].append(o[v][a:b]); S["out"][v]["detf"].append(sf.detection_features(dd, THR07[v], len(sids)))
                S["out"][v]["occ"].append(rk.slot_occlusion(len(sids), pairs, THR07[v], float(ASSOC[v]["overlap_thr"])))
            for key, th in (("fixable", THR07), ("fixable_iso", T_ISO)):
                wn = (o[LIGHT][a:b] >= th[LIGHT]) != gt; ws = (o[HEAVY][a:b] >= th[HEAVY]) != gt
                S[key].append(int((wn & ~ws & valid).sum()))
            r9 = ff[str(names[f])]
            S["B"].append(float(r9["brightness"]))
            S["M"].append(np.nan if r9["motion_raw"] in ("", "None") else float(sf.scale(float(r9["motion_raw"]), SCALES["motion"])))
        for k in ("B", "M", "fixable", "fixable_iso"):
            S[k] = np.array(S[k])
        seqs.append(S)
    return seqs

CACHE = OUT / "inputs_cache{}.pkl".format(SUF)
SEQ = None
if CACHE.exists():
    c = pickle.load(open(str(CACHE), "rb"))
    if c.get("hash") == CFG_HASH:
        SEQ = c["seq"]; print("inputs: cached")
if SEQ is None:
    t0 = time.time()
    try:
        SEQ = {s: build(s) for s in ("val", "test")}
    except Exception as e:
        write_status(STATUS, "13_end_to_end_evaluation", RUN_ID, "FAILED", CFG_HASH, repr(e)[:500]); raise
    pickle.dump({"hash": CFG_HASH, "seq": SEQ}, open(str(CACHE), "wb"))
    print("inputs built in {:.0f} s".format(time.time() - t0))
for s in SEQ:
    print("{:<5} sequences {} | frames {} | fixable slots nb07 {} / deployed threshold {}".format(s, len(SEQ[s]), sum(len(q["sec"]) for q in SEQ[s]),
          sum(int(q["fixable"].sum()) for q in SEQ[s]), sum(int(q["fixable_iso"].sum()) for q in SEQ[s])))

In [ ]:
# Runner and metrics: replay, evaluate, cost at C_FV/C_FO, transitions (false flips), per-sequence counts for the bootstrap
def ctx_for(pipe, wD=None):
    p = PIPES[pipe]
    return dict(BASECTX, thr=p["thr"], filter=p["filter"], fixable_key=p["fixable_key"], **({"wD": wD} if wD is not None else {}))

def transitions(seqs, outs):
    parts, slot_frames, slot_hours = [], 0, 0.0
    for q, o in zip(seqs, outs):
        T, S_ = len(q["sec"]), q["n_cols"]
        G = np.full((T, S_), -1, np.int8); P = np.full((T, S_), -1, np.int8)
        for t in range(T):
            vc = np.asarray(q["cols"][t])[q["valid"][t]]
            G[t, vc] = np.asarray(q["gt"][t])[q["valid"][t]]; P[t, vc] = o["pred"][t]
        parts.append(tf.transition_metrics(G, P, q["sec"], TOL_FRAMES))
        slot_frames += int((G >= 0).sum()); slot_hours += (q["sec"][-1] - q["sec"][0]) / 3600.0 * S_ if T > 1 else 0.0
    return tf.merge_transition_metrics(parts, slot_frames, slot_hours)

def run(split, strategy, cfg, pipe, wD=None, trans=True):
    ctx = ctx_for(pipe, wD)
    outs = [rp.run_sequence(q, strategy, cfg, ctx) for q in SEQ[split]]
    m = rp.evaluate(SEQ[split], outs, COST, cfg.get("max_age_s"))
    m["cost_per_1000_decisions"] = round(cs.cost_per_1000(m["FV"], m["FO"], m["slots"], C_FV, C_FO), 3)
    if trans:
        tr = transitions(SEQ[split], outs)
        m.update({"false_flips_per_1000": tr["false_per_1000_slot_frames"], "transition_detection": tr["detection_rate"],
                  "transition_delay_frames": tr["mean_delay_frames"]})
    per = []
    for q, o in zip(SEQ[split], outs):
        e = rp.evaluate([q], [o], COST)
        per.append(dict(e, cost=e["cost_per_frame"] * e["frames"]))
    m["_per_seq"] = bs.counts(per)
    return outs, m

KEYS = ("accuracy", "macro_f1", "FV", "FO", "cost_per_1000_decisions", "cost_per_frame", "skipped_share", "false_flips_per_1000", "transition_delay_frames")
HDR = "  {:<44} {:>8} {:>8} {:>6} {:>6} {:>9} {:>8} {:>7} {:>8} {:>7}".format("strategy | decision", "accuracy", "macroF1", "FV", "FO", "cost@{:g}".format(C_FV / C_FO), "compute", "skipped", "flips/1k", "delay")
def show(name, m):
    print("  {:<44} {:>8} {:>8} {:>6} {:>6} {:>9} {:>8} {:>7} {:>8} {:>7}".format(name, *[str(m.get(k)) for k in KEYS]))

STRATS = [("fixed YOLO11n", "fixed_n", {}), ("fixed YOLO11s", "fixed_s", {}), ("periodic YOLO11n c=2", "periodic_n", {"cadence": 2}),
          ("scene-triggered, no slot risk", "scene_only", CHOSEN["scene_only"]["accuracy_point"]),
          ("AdaptivePark (accuracy point)", "adaptive", CHOSEN["adaptive"]["accuracy_point"])]
if CHOSEN["adaptive"].get("efficiency_point"):
    STRATS.append(("AdaptivePark (efficiency point)", "adaptive", CHOSEN["adaptive"]["efficiency_point"]))
STRATS.append(("oracle (labels; upper bound)", "oracle", {}))

In [ ]:
# F1-F2: every strategy under both decision pipelines, VAL (reference) and TEST (reported)
RES, OUTS = {"val": {}, "test": {}}, {}
t0 = time.time()
try:
    for split in ("val", "test"):
        for pipe in ("nb07 rule", "deployed"):
            for name, strat, cfg in STRATS:
                o_, RES[split][(name, pipe)] = run(split, strat, cfg, pipe)
                if split == "test":
                    OUTS[(name, pipe)] = o_
except Exception as e:
    write_status(STATUS, "13_end_to_end_evaluation", RUN_ID, "FAILED", CFG_HASH, repr(e)[:500]); raise
print("{} runs in {:.0f} s".format(sum(len(r) for r in RES.values()), time.time() - t0))
for split in ("val", "test"):
    print("\n{} (accuracy over every frame; cost@{:g} = (C_FV FV + C_FO FO) per 1000 slot decisions; compute = YOLO11n units per frame, Kaggle T4 ratio)".format(
        split.upper() + (" - reported" if split == "test" else " - reference only, nothing is chosen here"), C_FV / C_FO))
    print(HDR)
    for pipe in ("nb07 rule", "deployed"):
        for name, _, _ in STRATS:
            show("{} | {}".format(name, pipe), RES[split][(name, pipe)])
a = RES["test"][("AdaptivePark (accuracy point)", "deployed")]
print("\nAdaptivePark, deployed rule, test | calls", a["calls"], "| mode share", a["mode_share"], "| reasons", a["reasons"], "| max-age violations", a["max_age_violations"])

In [ ]:
# F3: ablations of AdaptivePark (accuracy point) under the deployed rule, TEST. Same val-quantile rule for re-derived thresholds.
def val_quantile_D(wD, qd):
    outs = [rp.run_sequence(q, "fixed_n", {}, ctx_for("deployed", wD)) for q in SEQ["val"]]
    Dv = np.concatenate([o["D"] for o in outs])
    return None if qd is None else float(np.nanquantile(Dv, qd))

ACC = CHOSEN["adaptive"]["accuracy_point"]; QU = ACC.get("quantiles", {})
ABL = [("full AdaptivePark (deployed rule)", ACC, "deployed", None),
       ("- calibration (nb07 threshold + filter)", ACC, "filter only", None),
       ("- temporal filter (isotonic only)", ACC, "isotonic only", None),
       ("- calibration - filter (= nb07 rule)", ACC, "nb07 rule", None),
       ("- slot risk (= scene-triggered)", CHOSEN["scene_only"]["accuracy_point"], "deployed", None),
       ("- critical-slot refresh", dict(ACC, r_crit=None), "deployed", None),
       ("- max-age refresh", dict(ACC, max_age_s=None), "deployed", None)]
ZERO = []
for i, f in enumerate(sf.FEATURES):
    if W_D[i] <= 0:
        ZERO.append(f); continue
    w = W_D.copy(); w[i] = 0.0
    if w.sum() <= 0:
        continue
    w = w / w.sum()
    cfg = dict(ACC, d_hi=val_quantile_D(w, QU.get("d_hi")))
    if QU.get("lo") is not None and ACC.get("d_lo") is not None:
        cfg["d_lo"] = val_quantile_D(w, QU["lo"])
    ABL.append(("- D feature {} (weights re-normalised, d_hi = val q{})".format(f, QU.get("d_hi")), cfg, "deployed", w))
if CHOSEN["adaptive"].get("efficiency_point"):
    EFF = CHOSEN["adaptive"]["efficiency_point"]
    ABL += [("efficiency point (deployed rule)", EFF, "deployed", None), ("efficiency point - max-age refresh", dict(EFF, max_age_s=None), "deployed", None)]
ABLR = {}
try:
    for name, cfg, pipe, wD in ABL:
        _, ABLR[name] = run("test", "adaptive" if cfg.get("use_risk", True) else "scene_only", cfg, pipe, wD)
except Exception as e:
    write_status(STATUS, "13_end_to_end_evaluation", RUN_ID, "FAILED", CFG_HASH, repr(e)[:500]); raise
ref = ABLR["full AdaptivePark (deployed rule)"]
print("TEST ablations (delta vs full AdaptivePark, deployed rule)")
print(HDR + "  {:>9} {:>9}".format("d cost", "d compute"))
for name, m in ABLR.items():
    show(name, m)
    if name != "full AdaptivePark (deployed rule)":
        print("  {:>44}  d macro-F1 {:+.4f} | d cost@{:g} {:+.3f} | d compute {:+.3f}".format("", m["macro_f1"] - ref["macro_f1"], C_FV / C_FO,
              m["cost_per_1000_decisions"] - ref["cost_per_1000_decisions"], m["cost_per_frame"] - ref["cost_per_frame"]))
print("\nnot ablated (weight 0 in notebook 09's D, dropping them changes nothing):", ZERO or "none")
print("hysteresis: the chosen setting already has margin {} and dwell {}; no separate 'no hysteresis' run".format(ACC.get("margin"), ACC.get("min_dwell")))

In [ ]:
# Paired bootstrap over camera-day sequences (TEST): is the difference larger than sequence-to-sequence noise?
def get(name, pipe):
    return RES["test"][(name, pipe)]["_per_seq"]
AP = "AdaptivePark (accuracy point)"
PAIRS = [("AdaptivePark vs fixed YOLO11n (deployed rule)", get(AP, "deployed"), get("fixed YOLO11n", "deployed")),
         ("AdaptivePark vs fixed YOLO11s (deployed rule)", get(AP, "deployed"), get("fixed YOLO11s", "deployed")),
         ("AdaptivePark vs scene-triggered (deployed rule)", get(AP, "deployed"), get("scene-triggered, no slot risk", "deployed")),
         ("deployed vs nb07 rule (AdaptivePark)", get(AP, "deployed"), get(AP, "nb07 rule")),
         ("deployed vs nb07 rule (fixed YOLO11n)", get("fixed YOLO11n", "deployed"), get("fixed YOLO11n", "nb07 rule")),
         ("deployed vs nb07 rule (fixed YOLO11s)", get("fixed YOLO11s", "deployed"), get("fixed YOLO11s", "nb07 rule"))]
STATS = [("macro-F1", bs.macro_f1, 5), ("cost@{:g} per 1000".format(C_FV / C_FO), bs.cost_per_1000(C_FV, C_FO), 3), ("compute per frame", bs.compute_per_frame, 3)]
BOOT = {}
print("paired bootstrap, {} resamples of the {} test sequences: difference (A - B) [95% interval] share of resamples > 0".format(N_BOOT, len(SEQ["test"])))
for name, A, B in PAIRS:
    BOOT[name] = {}
    print("  " + name)
    for sname, f, nd in STATS:
        r = bs.paired(A, B, f, N_BOOT, seed=0); BOOT[name][sname] = r
        print("    {:<20} {:+.{p}f} [{:+.{p}f}, {:+.{p}f}]  > 0 in {:.1%}".format(sname, r["difference"], r["ci95"][0], r["ci95"][1], r["share_above_0"], p=nd))

In [ ]:
# Per site (TEST, deployed rule): accuracy, cost and compute of n, s and AdaptivePark on the frames of each site
def site_rows(name, pipe):
    acc = defaultdict(lambda: np.zeros(6))
    for q, o in zip(SEQ["test"], OUTS[(name, pipe)]):
        e = rp.evaluate([q], [o], COST)
        acc[q["site"]] += np.array([e["TP"], e["TN"], e["FO"], e["FV"], e["frames"], e["cost_per_frame"] * e["frames"]])
    return acc
SITES = {}
names = ("fixed YOLO11n", "fixed YOLO11s", AP)
rows = {n_: site_rows(n_, "deployed") for n_ in names}
print("{:<14} {:>7} | {:>8} {:>8} {:>8} | {:>8} {:>8} {:>8} | {:>7}".format("site", "frames", "acc n", "acc s", "acc AP", "cost n", "cost s", "cost AP", "comp AP"))
for site in sorted(rows[AP]):
    c = {n_: rows[n_][site] for n_ in names}
    SITES[site] = {n_: {"accuracy": round(bs.accuracy(c[n_]), 4), "macro_f1": round(bs.macro_f1(c[n_]), 4),
                        "cost_per_1000_decisions": round(bs.cost_per_1000(C_FV, C_FO)(c[n_]), 3), "compute_per_frame": round(bs.compute_per_frame(c[n_]), 3)} for n_ in names}
    print("{:<14} {:>7} | {:>8} {:>8} {:>8} | {:>8} {:>8} {:>8} | {:>7}".format(site, int(c[AP][4]), *[SITES[site][n_]["accuracy"] for n_ in names],
          *[SITES[site][n_]["cost_per_1000_decisions"] for n_ in names], SITES[site][AP]["compute_per_frame"]))

In [ ]:
# Plots: cost@ratio vs compute for every strategy and both rules; ablation deltas
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(15, 4.8))
for name, _, _ in STRATS:
    a0, a1 = RES["test"][(name, "nb07 rule")], RES["test"][(name, "deployed")]
    ax[0].annotate("", xy=(a1["cost_per_frame"], a1["cost_per_1000_decisions"]), xytext=(a0["cost_per_frame"], a0["cost_per_1000_decisions"]),
                   arrowprops={"arrowstyle": "->", "color": "0.6"})
    ax[0].scatter([a0["cost_per_frame"]], [a0["cost_per_1000_decisions"]], marker="o", facecolors="none", edgecolors="0.4")
    ax[0].scatter([a1["cost_per_frame"]], [a1["cost_per_1000_decisions"]], marker="o")
    ax[0].annotate(name, (a1["cost_per_frame"], a1["cost_per_1000_decisions"]), fontsize=7, xytext=(4, 3), textcoords="offset points")
ax[0].set(xlabel="compute per frame (YOLO11n units, Kaggle T4 ratio)", ylabel="cost per 1000 slot decisions (C_FV/C_FO = {:g})".format(C_FV / C_FO),
          title="TEST: open = nb07 rule, filled = deployed rule"); ax[0].grid(alpha=0.3)
nm = [n_ for n_ in ABLR if n_ != "full AdaptivePark (deployed rule)"]
d = [ABLR[n_]["cost_per_1000_decisions"] - ref["cost_per_1000_decisions"] for n_ in nm]
ax[1].barh(range(len(nm)), d); ax[1].set_yticks(range(len(nm))); ax[1].set_yticklabels(nm, fontsize=7); ax[1].axvline(0, color="k", lw=0.8)
ax[1].set(xlabel="change in cost per 1000 decisions vs full AdaptivePark (positive = worse)", title="TEST: ablations (deployed rule)"); ax[1].grid(alpha=0.3, axis="x")
plt.tight_layout(); plt.savefig(str(OUT / "evaluation{}.png".format(SUF)), dpi=110); plt.show()

In [ ]:
# Record, environment, status
from src.common.env_record import save_environment
def jclean(x):
    if isinstance(x, dict):
        return {str(k): jclean(v) for k, v in x.items() if not str(k).startswith("_")}
    if isinstance(x, (list, tuple)):
        return [jclean(v) for v in x]
    if isinstance(x, (float, np.floating)):
        return float(x) if np.isfinite(x) else None
    if isinstance(x, np.ndarray):
        return jclean(x.tolist())
    if isinstance(x, (np.integer, np.bool_)):
        return x.item()
    return x
RECORD = {"run_id": RUN_ID, "config_hash": CFG_HASH, "smoke_test": SMOKE, "dataset_split_hash": H, "notebook07_run_id": st07["run_id"],
          "notebook09_run_id": R09["run_id"], "notebook10_run_id": R10["run_id"], "notebook11_run_id": R11["run_id"],
          "decision": {"nb07_score_threshold": THR07, "deployed": {"calibrator": METH, "probability_threshold": P_THR, "score_equivalent": T_ISO,
                       "filter": FILT, "c_fv": C_FV, "c_fo": C_FO}},
          "compute_units": COST, "compute_note": "YOLO11s/YOLO11n Kaggle T4 batched ratio (notebook 06); NOT a Jetson measurement",
          "results": {split: {"{} | {}".format(*k): v for k, v in r.items()} for split, r in RES.items()},
          "ablations_test": ABLR, "ablation_rule": "thresholds re-derived as the same val quantile as notebook 10 chose; nothing else re-tuned",
          "zero_weight_features_not_ablated": ZERO, "bootstrap_test": BOOT, "sites_test_deployed": SITES,
          "semantics": "skipped frames show the last reliable (filtered) state; never a fresh vacant; accuracy over every frame",
          "limitation": "replay of cached detections; frames 5 min (PKLot) / 30 min (CNRPark-EXT) apart; compute is a Kaggle T4 ratio; bootstrap unit = camera-day sequence"}
(OUT / "evaluation_record{}.json".format(SUF)).write_text(json.dumps(jclean(RECORD), indent=2))
save_environment(str(OUT / "environment.json"))
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch" and CACHE.exists():
    CACHE.unlink()
write_status(STATUS, "13_end_to_end_evaluation", RUN_ID, "COMPLETE", CFG_HASH, "smoke={}".format(SMOKE),
             extra=jclean({k: {m: RES["test"][(k, "deployed")][m] for m in ("macro_f1", "cost_per_1000_decisions", "cost_per_frame")} for k, _, _ in STRATS}))
print("COMPLETE | saved to", OUT, sorted(p.name for p in OUT.iterdir()))

## What to send back
Download the notebook after **Save & Run All** (SMOKE = False) and send it.
`artifacts/evaluation/evaluation_record.json` feeds the final report and the dashboard's Experiments page, so **Save Version** with outputs.